# 00 - Supervised fine-tuning

Fine-tune Qwen3-0.6B on 256 GSM8K worked solutions and save one checkpoint for every RL notebook. The official test split remains held out.

In [ ]:
# Hyperparameters
base_model = "Qwen/Qwen3-0.6B"
dataset_id = "openai/gsm8k"
dataset_config = "main"
dataset_revision = "740312add88f781978c0658806c59bc2815b9866"
seed = 17
train_problems = 256
eval_problems = 64
num_epochs = 1
max_sequence_length = 1024
train_micro_batch_size = 2
gradient_accumulation_steps = 8
learning_rate = 2e-5
weight_decay = 0.0
warmup_ratio = 0.03
logging_steps = 1
eval_steps = 8
save_path = "./checkpoints/sft_base"
wandb_project = "swe-rl-ablations"
wandb_run_name = "sft-base"

effective_batch_size = train_micro_batch_size * gradient_accumulation_steps
assert train_problems == 256 and eval_problems == 64 and num_epochs == 1
assert train_problems % effective_batch_size == 0
print(f"Effective SFT batch size: {effective_batch_size}")

In [ ]:
from pathlib import Path
import os
import re

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer

project_root = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
checkpoint_path = project_root / save_path
os.environ["WANDB_PROJECT"] = wandb_project

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this project.")
try:
    import flash_attn  # noqa: F401
except ImportError as exc:
    raise RuntimeError("Install flash-attn before running the notebook.") from exc
set_seed(seed)

In [ ]:
train_rows = load_dataset(dataset_id, dataset_config, split="train", revision=dataset_revision)
eval_rows = load_dataset(dataset_id, dataset_config, split="test", revision=dataset_revision)
if set(train_rows.column_names) != {"question", "answer"}:
    raise RuntimeError(f"Unexpected GSM8K schema: {train_rows.column_names}")
train_rows = train_rows.shuffle(seed=seed).select(range(train_problems))
eval_rows = eval_rows.shuffle(seed=seed).select(range(eval_problems))
print(f"SFT train: {len(train_rows)} | held-out test: {len(eval_rows)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def assistant_solution(answer):
    match = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", answer)
    if not match:
        raise ValueError(f"Malformed GSM8K answer: {answer!r}")
    reasoning = answer[:match.start()].replace("<<", "(").replace(">>", ")").strip()
    number = match.group(1)
    return f"{reasoning}\nFinal answer: {number}"


def conversation(row):
    messages = [
        {
            "role": "user",
            "content": "Solve this grade-school math problem. Show your reasoning, then end with `Final answer: <number>`.\n\n" + row["question"],
        },
        {"role": "assistant", "content": assistant_solution(row["answer"])},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_text = train_rows.map(conversation, remove_columns=train_rows.column_names)
eval_text = eval_rows.map(conversation, remove_columns=eval_rows.column_names)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
model.config.use_cache = False

training_args = SFTConfig(
    output_dir=str(checkpoint_path),
    run_name=wandb_run_name,
    dataset_text_field="text",
    max_length=max_sequence_length,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=train_micro_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    warmup_ratio=warmup_ratio,
    logging_steps=logging_steps,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="epoch",
    report_to="wandb",
    bf16=True,
    gradient_checkpointing=True,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_text,
    eval_dataset=eval_text,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(str(checkpoint_path))
tokenizer.save_pretrained(str(checkpoint_path))
assert (checkpoint_path / "config.json").exists()
print(f"Saved {checkpoint_path}")